# Polymarket FX Monitor — Walkthrough

This notebook walks through the **Polymarket FX monitoring pipeline**: fetch active prediction markets, filter by FX-related keywords, score relevance with Claude, store snapshots in SQLite, and generate a markdown report.

**Pipeline steps:**
1. **Fetch** — Get active (open) binary markets from the Polymarket Gamma API, ordered by 24h volume
2. **Keyword filter** — Keep only markets whose question/description match FX keywords (central banks, macro, currencies, etc.)
3. **Classify** — Claude scores each market 0–10 for FX relevance and lists affected currency pairs
4. **Store** — Save scored markets as a snapshot run in SQLite; compute deltas vs previous run
5. **Report** — Generate markdown: summary stats, biggest movers, high relevance, watchlist, low relevance

**Requirements:** `ANTHROPIC_API_KEY` in `.env` for classification. Fetch and keyword filter need no API keys.

## Setup

1. **Select the project kernel**: click the kernel name (top right) → **Select Another Kernel** → choose **"Python (Experimental-Sandbox .venv)"** (or `Experimental-Sandbox/.venv/bin/python`).
2. Run the cell below to set project root, load `.env`, and import the monitor modules.

In [ ]:
import os
import sys
from pathlib import Path

# Find project root. Use kernel ".venv (Experimental-Sandbox)" so deps (dotenv, etc.) are available.
ROOT = Path.cwd()
if (ROOT / "Experimental-Sandbox" / "polymarket_monitor").is_dir():
    ROOT = ROOT / "Experimental-Sandbox"
elif (ROOT / "polymarket_monitor").is_dir():
    pass
elif ROOT.name == "notebooks" and (ROOT.parent / "polymarket_monitor").is_dir():
    ROOT = ROOT.parent
else:
    for parent in ROOT.parents:
        if (parent / "polymarket_monitor").is_dir():
            ROOT = parent
            break
        if (parent / "Experimental-Sandbox" / "polymarket_monitor").is_dir():
            ROOT = parent / "Experimental-Sandbox"
            break
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
if not os.environ.get("ANTHROPIC_API_KEY") and (ROOT.parent / ".env").exists():
    load_dotenv(ROOT.parent / ".env")

from polymarket_monitor.fetcher import fetch_active_markets, RawMarket
from polymarket_monitor.keywords import filter_markets, FX_KEYWORDS, FilteredMarket
from polymarket_monitor.classifier import classify_markets, ScoredMarket
from polymarket_monitor.db import SnapshotDB, MarketSnapshot
from polymarket_monitor.report import generate_report

print("Project root:", ROOT)
print("ANTHROPIC_API_KEY set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

---
## Step 1: Fetch active markets

**fetcher** calls the Polymarket Gamma API: `https://gamma-api.polymarket.com/markets` with `closed=false`, ordered by 24h volume descending. Results are paginated (up to 10 pages × 100). Only **binary** markets (`outcome_count == 2`) are kept.

Each **RawMarket** has: `market_id`, `slug`, `question`, `description`, `yes_price`, `no_price`, `volume_24h`, `liquidity`, and a `url` property.

In [ ]:
raw_markets = fetch_active_markets()

print(f"Fetched {len(raw_markets)} binary markets")
if raw_markets:
    m = raw_markets[0]
    print("\nExample (first by volume):")
    print("  question:", m.question[:80] + "..." if len(m.question) > 80 else m.question)
    print("  yes_price:", m.yes_price)
    print("  volume_24h:", m.volume_24h)
    print("  url:", m.url)

---
## Step 2: Keyword filter

**keywords** applies a regex over `question` and `description` using **FX_KEYWORDS** (central_banks, macro, trade, geopolitics, currencies, fiscal). Markets that match at least one keyword become **FilteredMarket** with `matched_categories` and `matched_keywords`.

In [ ]:
print("FX_KEYWORDS categories:", list(FX_KEYWORDS.keys()))
print()
filtered = filter_markets(raw_markets)
print(f"After keyword filter: {len(filtered)} markets")
if filtered:
    fm = filtered[0]
    print("\nExample:")
    print("  question:", fm.market.question[:70] + "..." if len(fm.market.question) > 70 else fm.market.question)
    print("  matched_categories:", fm.matched_categories)
    print("  matched_keywords (sample):", fm.matched_keywords[:8])

---
## Step 3: Claude FX classification

**classifier** sends batches of filtered markets to Claude. For each market it returns:
- **fx_score** (0–10): 9–10 = direct FX; 7–8 = strong FX impact; 4–6 = indirect; 1–3 = weak; 0 = none
- **fx_reason**: one-sentence explanation
- **affected_pairs**: e.g. `["EUR/USD", "USD/JPY"]`

Requires **ANTHROPIC_API_KEY**. On batch failure, markets in that batch get a default score of 5.

In [ ]:
anthropic_key = os.environ.get("ANTHROPIC_API_KEY")

if not anthropic_key:
    print("ANTHROPIC_API_KEY not set — skip classification. Use dry-run report or set key to run full pipeline.")
    scored = []
else:
    scored = classify_markets(filtered, anthropic_key)
    print(f"Classified {len(scored)} markets")
    if scored:
        s = scored[0]
        print("\nExample ScoredMarket:")
        print("  question:", s.market.market.question[:60] + "..." if len(s.market.market.question) > 60 else s.market.market.question)
        print("  fx_score:", s.fx_score)
        print("  fx_reason:", s.fx_reason)
        print("  affected_pairs:", s.affected_pairs)

---
## Step 4: SQLite snapshot storage and deltas

**db.SnapshotDB** stores each run in `runs` (run_id, fetched_at) and each market in `snapshots` (run_id, market fields, fx_score, fx_reason, affected_pairs, matched_categories).

**get_snapshots_with_deltas()** returns the latest run with **yes_price_delta** and **volume_delta** vs the previous run (or `None` for first run / new markets).

In [ ]:
db_path = ROOT / "data" / "polymarket_notebook.db"
db_path.parent.mkdir(parents=True, exist_ok=True)

if scored:
    db = SnapshotDB(str(db_path))
    try:
        run_id = db.store_snapshot(scored)
        print(f"Stored snapshot as run_id={run_id}")
        snapshots = db.get_snapshots_with_deltas()
        print(f"Retrieved {len(snapshots)} snapshots with deltas")
        if snapshots:
            s = snapshots[0]
            print("\nExample MarketSnapshot:")
            print("  question:", s.question[:55] + "..." if len(s.question) > 55 else s.question)
            print("  yes_price:", s.yes_price, "|", "yes_price_delta:", s.yes_price_delta)
            print("  fx_score:", s.fx_score)
    finally:
        db.close()
else:
    print("No scored markets — skipping DB. Run Step 3 with ANTHROPIC_API_KEY to populate.")
    snapshots = []

---
## Step 5: Generate markdown report

**report.generate_report** builds a structured report:
- **Summary** — count of markets, high FX relevance (score ≥ 7), markets with price history
- **Biggest Movers** — top 10 by |yes_price_delta|
- **High FX Relevance (Score ≥ 7)** — full details per market
- **Watchlist (Score 4–6)** — table
- **Low relevance (score < 4)** — collapsible `<details>` block

If we didn't run classification, we can still build a **dry-run** style report from filtered markets only (no scores). Below we generate the full report when we have snapshots.

In [ ]:
if snapshots:
    report_md = generate_report(snapshots)
    print("Report length:", len(report_md), "chars")
    print("\n--- Report preview (first 120 lines) ---\n")
    print("\n".join(report_md.split("\n")[:120]))
else:
    # Minimal report when no snapshots (e.g. no API key)
    report_md = "# Polymarket FX Monitor\n\nNo snapshots. Run classification (Step 3) with ANTHROPIC_API_KEY and then Step 4.\n"
    print(report_md)

---
## Running the full pipeline from the CLI

From the project root:

```bash
python run_polymarket.py --output data/polymarket_fx_report.md --db data/polymarket.db -v
```

**Dry run** (fetch + keyword filter only, no Claude or DB):

```bash
python run_polymarket.py --dry-run -v
```

Install and run via entry point:

```bash
pip install -e .
polymarket-monitor --output data/polymarket_fx_report.md -v
```